# MIT-BIH Arrhythmia Database Exploration

Load and analyze the MIT-BIH arrhythmia database for ECG anomaly detection.

In [1]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import wfdb

# Add parent directory to path
sys.path.insert(0, '..')
from utils import (load_mit_bih_record, load_all_mit_bih_records, 
                   extract_ecg_features, normalize_signal, plot_ecg,
                   find_r_peaks, segment_signal_by_beats, create_beat_dataset)

# Set up plotting
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)

## Load MIT-BIH Database

In [2]:
# Path to MIT-BIH database
db_path = 'C:\\Users\\Admin\\Downloads\\mit-bih-arrhythmia-database-1.0.0'

# Load all records (this might take a moment)
print("Loading MIT-BIH database...")
records = load_all_mit_bih_records(db_path)
print(f"\nLoaded {len(records)} records")

Loading MIT-BIH database...
Loading record 100...
Error loading record 100: 'n_samples'
Loading record 101...
Error loading record 101: 'n_samples'
Loading record 102...
Error loading record 102: 'n_samples'
Loading record 103...
Error loading record 103: 'n_samples'
Loading record 104...
Error loading record 104: 'n_samples'
Loading record 105...
Error loading record 105: 'n_samples'
Loading record 106...
Error loading record 106: 'n_samples'
Loading record 107...
Error loading record 107: 'n_samples'
Loading record 108...
Error loading record 108: 'n_samples'
Loading record 109...
Error loading record 109: 'n_samples'
Loading record 111...
Error loading record 111: 'n_samples'
Loading record 112...
Error loading record 112: 'n_samples'
Loading record 113...
Error loading record 113: 'n_samples'
Loading record 114...
Error loading record 114: 'n_samples'
Loading record 115...
Error loading record 115: 'n_samples'
Loading record 116...
Error loading record 116: 'n_samples'
Loading reco

## Record Overview

In [3]:
from config import FS, HALF_WINDOW, BEAT_LEN

# Test on first record
if records:
    record = records[0]
    signal = record['signal'][:, 0]
    
    # Find R-peaks
    r_peaks = find_r_peaks(signal, fs=FS)
    print(f"Found {len(r_peaks)} R-peaks in record {record['record']}")
    print(f"R-peak indices (first 10): {r_peaks[:10]}")
    
    # Plot signal with detected R-peaks
    plt.figure(figsize=(14, 5))
    time_axis = np.arange(len(signal)) / FS
    
    plt.plot(time_axis, signal, lw=0.5, color='steelblue', label='ECG Signal')
    plt.plot(time_axis[r_peaks], signal[r_peaks], 'ro', markersize=6, label='Detected R-peaks')
    
    # Also plot annotations
    for sample in record['anno_sample'][:20]:
        plt.axvline(sample / FS, color='green', alpha=0.3, linewidth=0.5)
    
    plt.xlabel('Time (seconds)')
    plt.ylabel('Amplitude (mV)')
    plt.title(f'Record {record["record"]} - R-peak Detection')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [4]:
if records:
    # Create dataset from all records (use first 5 for quick test)
    print("Creating beat dataset...")
    dataset_df = create_beat_dataset(records[:5], fs=FS, normalize=True)
    
    print(f"\nDataset created:")
    print(f"  Total beats: {len(dataset_df)}")
    print(f"  Shape per beat: {len(dataset_df.iloc[0]['signal'])} samples")
    print(f"\nLabel distribution:")
    print(dataset_df['label'].value_counts())
    print(f"\nRecords in dataset: {dataset_df['record'].unique()}")
    
    # Visualize distribution
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Label distribution
    label_counts = dataset_df['label'].value_counts()
    axes[0].bar(label_counts.index, label_counts.values, color=['steelblue', 'coral'])
    axes[0].set_title('Label Distribution')
    axes[0].set_ylabel('Count')
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Records distribution
    record_counts = dataset_df['record'].value_counts().sort_index()
    axes[1].bar(record_counts.index, record_counts.values, color='steelblue')
    axes[1].set_title('Beats per Record')
    axes[1].set_xlabel('Record')
    axes[1].set_ylabel('Count')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    # Show sample beats of each class
    print("\nSample beats visualization:")
    normal_beats = dataset_df[dataset_df['label'] == 'normal']['signal'].values[:3]
    anomaly_beats = dataset_df[dataset_df['label'] == 'anomaly']['signal'].values[:3]
    
    fig, axes = plt.subplots(2, 3, figsize=(14, 6))
    
    for i, beat in enumerate(normal_beats):
        time_beat = np.arange(len(beat)) / FS
        axes[0, i].plot(time_beat, beat, lw=1.5, color='steelblue')
        axes[0, i].set_title(f'Normal Beat {i+1}')
        axes[0, i].grid(True, alpha=0.3)
    
    for i, beat in enumerate(anomaly_beats):
        time_beat = np.arange(len(beat)) / FS
        axes[1, i].plot(time_beat, beat, lw=1.5, color='coral')
        axes[1, i].set_title(f'Anomaly Beat {i+1}')
        axes[1, i].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Create Full Beat Dataset

In [5]:
if records and len(r_peaks) > 0:
    # Segment into beats
    beats, valid_indices = segment_signal_by_beats(signal, r_peaks, fs=FS, half_window=HALF_WINDOW)
    
    print(f"Total R-peaks: {len(r_peaks)}")
    print(f"Valid beats (within bounds): {len(beats)}")
    print(f"Beat length: {BEAT_LEN} samples ({BEAT_LEN/FS:.3f} seconds)")
    
    # Visualize first 5 beats
    fig, axes = plt.subplots(5, 1, figsize=(12, 10))
    
    for i in range(min(5, len(beats))):
        beat = beats[i]
        time_beat = np.arange(len(beat)) / FS
        
        axes[i].plot(time_beat, beat, lw=1.5, color='steelblue')
        axes[i].axvline(HALF_WINDOW/FS, color='red', linestyle='--', alpha=0.7, label='R-peak')
        axes[i].set_title(f'Beat {i+1} (R-peak at {HALF_WINDOW/FS:.3f}s)')
        axes[i].set_ylabel('Amplitude (mV)')
        axes[i].grid(True, alpha=0.3)
        axes[i].legend()
    
    axes[-1].set_xlabel('Time (seconds)')
    plt.tight_layout()
    plt.show()

## Beat Segmentation

## Test R-Peak Detection and Beat Segmentation

In [6]:
# Display information about loaded records
if records:
    print("Record Details:")
    for record in records[:5]:  # Show first 5 records
        print(f"\nRecord {record['record']}:")
        print(f"  Channels: {record['n_sig']} ({', '.join(record['sig_name'])})")
        print(f"  Sampling Rate: {record['fs']} Hz")
        print(f"  Samples: {record['n_samples']}")
        print(f"  Duration: {record['n_samples'] / record['fs']:.1f} seconds")
        print(f"  Annotations: {len(record['anno_sample'])} events")
        print(f"  Annotation types: {set(record['anno_symbol'])}")

## Visualize Sample Records

In [7]:
if records:
    # Plot first 3 records
    fig, axes = plt.subplots(3, 1, figsize=(14, 8))
    
    for idx in range(min(3, len(records))):
        record = records[idx]
        signal = record['signal'][:, 0]  # Use first channel
        fs = record['fs']
        
        time_axis = np.arange(len(signal)) / fs
        
        axes[idx].plot(time_axis, signal, lw=0.5, color='steelblue')
        axes[idx].set_title(f"Record {record['record']} - {record['sig_name'][0]}")
        axes[idx].set_xlabel('Time (seconds)')
        axes[idx].set_ylabel('Amplitude (mV)')
        
        # Mark annotations
        for sample, symbol in zip(record['anno_sample'], record['anno_symbol']):
            if sample < len(signal):
                axes[idx].axvline(sample / fs, color='red', alpha=0.3, linewidth=0.5)
        
        axes[idx].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Annotation Statistics

In [8]:
if records:
    # Collect all annotations
    all_annotations = []
    
    for record in records:
        for symbol in record['anno_symbol']:
            all_annotations.append(symbol)
    
    # Count annotations
    from collections import Counter
    anno_counts = Counter(all_annotations)
    
    print("Annotation Types and Counts:")
    for anno, count in sorted(anno_counts.items(), key=lambda x: x[1], reverse=True):
        pct = (count / len(all_annotations)) * 100
        print(f"  {anno}: {count} ({pct:.1f}%)")
    
    # Plot annotation distribution
    fig, ax = plt.subplots(figsize=(10, 6))
    annotations = list(anno_counts.keys())
    counts = list(anno_counts.values())
    
    colors = plt.cm.Set3(np.linspace(0, 1, len(annotations)))
    ax.bar(annotations, counts, color=colors, edgecolor='black')
    ax.set_title('Distribution of Annotation Types in MIT-BIH Database')
    ax.set_xlabel('Annotation Type')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

## Extract Features from Signals

In [9]:
if records:
    # Extract features for each record
    features_list = []
    
    for record in records:
        signal = record['signal'][:, 0]  # First channel
        features = extract_ecg_features(signal, record['fs'])
        features['record'] = record['record']
        features['n_anno'] = len(record['anno_sample'])
        features_list.append(features)
    
    features_df = pd.DataFrame(features_list)
    
    print("Feature Summary:")
    print(features_df.describe())

## Zoom in on Specific Event

In [10]:
if records:
    # Pick a record with annotations
    record = records[0]
    signal = record['signal'][:, 0]
    fs = record['fs']
    
    # Get first annotation
    if len(record['anno_sample']) > 0:
        event_sample = record['anno_sample'][0]
        event_symbol = record['anno_symbol'][0]
        
        # Extract window around event (2 seconds before and after)
        window_samples = fs * 2
        start = max(0, event_sample - window_samples)
        end = min(len(signal), event_sample + window_samples)
        
        window_signal = signal[start:end]
        time_window = np.arange(len(window_signal)) / fs
        
        plt.figure(figsize=(14, 5))
        plt.plot(time_window, window_signal, lw=1.5, color='steelblue')
        plt.axvline(2, color='red', linestyle='--', label=f'Event: {event_symbol}')
        plt.title(f'Record {record["record"]} - Event Detail View')
        plt.xlabel('Time (seconds, relative to event)')
        plt.ylabel('Amplitude (mV)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

## Signal Normalization & Preprocessing

In [11]:
if records:
    record = records[0]
    signal = record['signal'][:, 0]
    
    # Normalize signal
    normalized = normalize_signal(signal)
    
    # Create comparison plot
    fig, axes = plt.subplots(2, 1, figsize=(14, 6))
    
    # Raw signal
    axes[0].plot(signal[:1000], lw=1, color='steelblue')
    axes[0].set_title('Raw ECG Signal (First 1000 samples)')
    axes[0].set_ylabel('Amplitude (mV)')
    axes[0].grid(True, alpha=0.3)
    
    # Normalized signal
    axes[1].plot(normalized[:1000], lw=1, color='darkgreen')
    axes[1].set_title('Normalized ECG Signal (0-1 range)')
    axes[1].set_xlabel('Samples')
    axes[1].set_ylabel('Normalized Amplitude')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Create Training Dataset

In [12]:
if records:
    # Create a training dataset with normalized signals and labels
    training_data = []
    
    window_size = 256  # 256 samples per window (~1 second at 256 Hz)
    step_size = 128    # Overlap windows by 50%
    
    for record in records[:5]:  # Use first 5 records for now
        signal = record['signal'][:, 0]
        fs = record['fs']
        
        # Normalize
        signal_norm = normalize_signal(signal)
        
        # Create windows
        for start in range(0, len(signal_norm) - window_size, step_size):
            window = signal_norm[start:start + window_size]
            
            # Check if window contains annotation
            window_end = start + window_size
            has_anomaly = any(start <= sample < window_end for sample in record['anno_sample'])
            
            training_data.append({
                'record': record['record'],
                'signal': window,
                'label': 'anomaly' if has_anomaly else 'normal'
            })
    
    print(f"Created {len(training_data)} windows")
    
    # Count labels
    labels_count = pd.Series([d['label'] for d in training_data]).value_counts()
    print(f"\nLabel distribution:\n{labels_count}")

## Summary

In [13]:
print("="*60)
print("MIT-BIH DATABASE EXPLORATION SUMMARY")
print("="*60)
if records:
    print(f"Total records loaded: {len(records)}")
    print(f"Sampling rates: {set(r['fs'] for r in records)}")
    print(f"Record durations: {min(r['n_samples']/r['fs'] for r in records):.0f}s to {max(r['n_samples']/r['fs'] for r in records):.0f}s")
    print(f"Total annotations: {sum(len(r['anno_sample']) for r in records)}")
    print(f"\nNext steps:")
    print("1. Create full training dataset with windowing")
    print("2. Train anomaly detection models (Autoencoder, VAE, LSTM, LNN)")
    print("3. Evaluate model performance on test set")
    print("4. Compare interpretability across models")
print("="*60)

MIT-BIH DATABASE EXPLORATION SUMMARY
